# BIQA CNN — ocena jakości zdjęć po zmianie rozdzielczości

Model wzorowany na **Frackiewicz, Palus, Trojanowski — *Blind Image Quality Assessment
Using CNNs* (Sensors 2025, 25, 7078)**, ale z jedną kluczową modyfikacją: **wejście to
cały obraz 256×256**, nie patche 32×32.

### Dlaczego nie patche?
Papier używa patchy 32×32, bo zakłada **homogeniczną degradację** w obrazie (szum
Gaussa, JPEG, kontrast — eq. ze stałymi parametrami w każdym pikselu). U nas degradacja
to **downscale → upscale**, gdzie:

- ESRGAN/Real-ESRGAN halucynuje detale inaczej na krawędziach niż na płaskich obszarach,
- box filter rozmywa tekstury inaczej w zależności od ich częstotliwości,
- aliasing po NN ujawnia się tylko na obszarach z wysokimi częstotliwościami.

Patch 32×32 z gładkiego nieba nie odróżni Real-ESRGAN od bicubic. Pełen kontekst tak.

### Co zostaje z papieru
1. **Lokalna normalizacja MSCN** (równ. 1–3) — niezmienione.
2. **Bloki Conv→BN→MaxPool** z progresją kanałów 32→64→128→256 — rozszerzone do 6 bloków,
   żeby zejść z 256×256 do 4×4 (Flatten = 4096 wartości).
3. **MAE loss** (równ. 5), wyjście liniowe, Adam.
4. **Optuna + TPE** (Tabela 2: n_neurons ∈ [500,1000], lr ∈ [0.001,0.01], dropout ∈ [0,0.8]).
5. **Metryki**: PLCC (z dopasowaniem 5-parametrowej logistic, równ. 6), SROCC, KROCC.

### Co dochodzi
Splity z manifestu (`train` / `val` / `test`) — deterministyczne, bez przecieków.
Heurystyczny pseudo-MOS z parametrów degradacji (manifest nie ma gotowych metryk FR).


## 1. Importy

In [ ]:
import os, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

from PIL import Image
from scipy.ndimage import uniform_filter
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr, kendalltau

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"TensorFlow: {tf.__version__}")
print(f"Optuna:     {optuna.__version__}")
print(f"GPU:        {tf.config.list_physical_devices('GPU') or 'CPU only'}")

## 2. Konfiguracja

Jedyne miejsce do ręcznej edycji. Ścieżki — pod swój dysk.

In [ ]:
# --- Ścieżki ---
DATA_ROOT     = Path('/media/myDisk/1K1K_po_zmianie_rozdzielczosci')   # gdzie leżą PNG-i
MANIFEST_PATH = Path('manifest.jsonl')                                 # plik z pipeline'u
OUT_DIR       = Path('out');  OUT_DIR.mkdir(exist_ok=True)

# --- Dane ---
IMG_SIZE    = 256              # zgodne z target_size_px w pipelinie
NORM_KERNEL = 3                # okno do lokalnej normalizacji (równ. 1-3 z papieru)
NORM_C      = 1.0 / 255.0      # stabilizator równania 1

# --- Trening ---
N_OPTUNA_TRIALS = 20
EPOCHS_TUNING   = 25           # krótszy trening w czasie tuningu
EPOCHS_FINAL    = 60           # dłuższy trening finalny (i tak utnie early stopping)
PATIENCE        = 8
BATCH_SIZE      = 16           # obrazy 256×256 — zostaw mały batch, żeby weszło na GPU

# --- Zakresy hiperparametrów do Optuny (Tabela 2 z papieru) ---
HP_SPACE = {
    'n_neurons':    (500, 1000),
    'eta':          (1e-3, 1e-2),
    'dropout_rate': (0.0,  0.8),
}

# --- Reprodukowalność ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"DATA_ROOT     = {DATA_ROOT}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")
print(f"IMG_SIZE      = {IMG_SIZE}")

## 3. Pseudo-MOS — etykieta uczenia

Manifest **nie zawiera** gotowych metryk FR (PSNR / SSIM / LPIPS / pseudo_mos), mimo że
pipeline miał je liczyć — to chyba wcześniejsza wersja. Dlatego cel uczenia wyciągamy
z parametrów degradacji:

- `CTRL` (oryginał) → 100
- kara za `scale_factor` (im więcej upchnęliśmy w dół, tym trudniej zrekonstruować)
- jakość pary (downscale, upscale) — Real-ESRGAN rekonstruuje dużo, BOX rozmywa, NN aliasuje, itd.

> **Hook do podmiany:** jeśli dodasz do manifestu prawdziwe pseudo_mos (np. fuzję
> PSNR + SSIM + LPIPS), wystarczy zmienić `pseudo_mos_from_entry()` na
> `return float(entry['metrics']['pseudo_mos'])`.


In [ ]:
# Bazowa jakość pary (downscale_abbr, upscale_abbr) przy scale_factor = 8
_BASE_QUALITY_AT_SF8 = {
    ('CTRL',    'CTRL'):    100.0,
    ('BIC',     'RESRGAN'):  80.0,   # SOTA SR rekonstruuje detale
    ('LANCZOS', 'ESRGAN'):   70.0,   # klasyczny ESRGAN — mocny, ale halucynuje
    ('NN',      'BIC'):      45.0,   # NN aliasuje, BIC tylko wygładza
    ('BOX',     'BIC'):      40.0,   # box filter najmocniej traci w dół
}

# Kara za większy scale_factor (sf=8 referencja)
_SF_PENALTY = {8: 0.0, 16: 12.0, 32: 25.0}


def pseudo_mos_from_entry(entry: dict) -> float:
    """Zwraca pseudo-MOS w [0, 100] na podstawie pól manifestu."""
    deg = entry['degradation']
    if deg['downscale']['abbr'] == 'CTRL':
        return 100.0

    key  = (deg['downscale']['abbr'], deg['upscale']['abbr'])
    base = _BASE_QUALITY_AT_SF8.get(key, 50.0)
    pen  = _SF_PENALTY.get(deg['scale_factor'], 0.0)
    return float(np.clip(base - pen, 0.0, 100.0))


# Sanity check
for d in [('CTRL','CTRL',8), ('BIC','RESRGAN',8), ('BIC','RESRGAN',32),
          ('LANCZOS','ESRGAN',16), ('BOX','BIC',32), ('NN','BIC',8)]:
    e = {'degradation': {'downscale': {'abbr': d[0]}, 'upscale': {'abbr': d[1]}, 'scale_factor': d[2]}}
    print(f"  {d[0]:>8s} → {d[1]:<8s}  sf={d[2]:<2d}  → MOS = {pseudo_mos_from_entry(e):5.1f}")

## 4. Lokalna normalizacja (równ. 1–3 z papieru)

$$\hat I(x,y) = \frac{I(x,y) - \mu(x,y)}{\sigma(x,y) + C}$$

gdzie $\mu, \sigma$ liczone w oknie 3×3 z wagami $\omega(i,j) = 1/9$.

Wariancja przez tożsamość $\sigma^2 = E[I^2] - \mu^2$ — wektorowo, per-kanał RGB.


In [ ]:
def local_normalize(img: np.ndarray, kernel: int = NORM_KERNEL, c: float = NORM_C) -> np.ndarray:
    """
    MSCN (równ. 1-3). img: (H, W, 3) float32 w [0, 1]. Zwraca to samo shape.
    """
    img = img.astype(np.float32, copy=False)
    out = np.empty_like(img)
    for ch in range(img.shape[2]):
        I   = img[..., ch]
        mu  = uniform_filter(I,     size=kernel, mode='reflect')
        mu2 = uniform_filter(I * I, size=kernel, mode='reflect')
        var = np.maximum(mu2 - mu * mu, 0.0)
        out[..., ch] = (I - mu) / (np.sqrt(var) + c)
    return out


# Sanity check
_dummy = np.random.rand(IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
_norm  = local_normalize(_dummy)
print(f"Test: shape={_norm.shape}, mean={_norm.mean():+.4f}, std={_norm.std():.4f}")

## 5. Wczytywanie pojedynczego obrazu

Cały obraz 256×256 → `/255` → lokalna normalizacja. Bez patchowania — to klucz tej wersji.


In [ ]:
def load_image(image_path) -> np.ndarray:
    """Wczytaj obraz → (256, 256, 3) po lokalnej normalizacji."""
    img = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.float32) / 255.0
    if img.shape[:2] != (IMG_SIZE, IMG_SIZE):
        img = np.asarray(Image.fromarray((img * 255).astype(np.uint8))
                              .resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS),
                         dtype=np.float32) / 255.0
    return local_normalize(img)


# Test na pierwszym znalezionym pliku
_test_path = next(DATA_ROOT.rglob('*.png'), None)
if _test_path is not None:
    _arr = load_image(_test_path)
    print(f"OK: {_test_path.name}  →  {_arr.shape}, dtype={_arr.dtype}")
else:
    print(f"⚠ Brak .png w {DATA_ROOT} — sprawdź DATA_ROOT przed treningiem.")

## 6. Wczytanie manifestu + split

Splity bierzemy bezpośrednio z manifestu — są deterministyczne i bez przecieków obrazu
źródłowego (`source_image_id`) między zbiorami.


In [ ]:
def load_manifest(path) -> pd.DataFrame:
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            if e.get('is_stress_test'):
                continue
            deg = e['degradation']
            rows.append({
                'sample_id':    e['sample_id'],
                'source_id':    e['source_image_id'],
                'path':         DATA_ROOT / e['file']['path_relative'],
                'split':        e['split'],
                'mos':          pseudo_mos_from_entry(e),
                'pair_abbr':    f"{deg['downscale']['abbr']}→{deg['upscale']['abbr']}",
                'scale_factor': deg['scale_factor'],
            })
    return pd.DataFrame(rows)


df = load_manifest(MANIFEST_PATH)
print(f"Załadowano {len(df)} próbek\n")
print("Splity:")
print(df['split'].value_counts().to_string(), '\n')
print("Rozkład MOS po splitach:")
print(df.groupby('split')['mos'].describe()[['count', 'mean', 'std', 'min', 'max']].round(2), '\n')
print("Pary degradacji:")
print(df['pair_abbr'].value_counts().to_string())

## 7. Budowa tablic numpy

Jeden obraz = jedna próbka (bez patchy). Ładujemy wszystko do RAM-u —
1260 × 256 × 256 × 3 × 4B ≈ **990 MB**. Mieści się; jeśli nie, podmień na `tf.data.Dataset`
z lazy loadingiem (kod poniżej w komentarzu).


In [ ]:
def build_arrays(df_subset: pd.DataFrame, name: str = '') -> tuple:
    """Zwraca: X (N, 256, 256, 3), y (N,)."""
    df_subset = df_subset.reset_index(drop=True)
    N = len(df_subset)
    X = np.empty((N, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    y = np.empty(N, dtype=np.float32)

    for i, row in df_subset.iterrows():
        X[i] = load_image(row['path'])
        y[i] = row['mos']
        if (i + 1) % 100 == 0:
            print(f"  [{name}] {i+1}/{N}")

    print(f"[{name}] X={X.shape}  ({X.nbytes / 1e9:.2f} GB),  y={y.shape}")
    return X, y


X_train, y_train = build_arrays(df[df['split'] == 'train'], 'train')
X_val,   y_val   = build_arrays(df[df['split'] == 'val'],   'val')
X_test,  y_test  = build_arrays(df[df['split'] == 'test'],  'test')

# Trzymamy też df_test, żeby przy ewaluacji rozbić wyniki na pary degradacji
df_test = df[df['split'] == 'test'].reset_index(drop=True)

## 8. Architektura — 6 bloków konwolucyjnych

Wzorowane na Tabeli 1 z papieru, ale rozszerzone żeby przyjąć 256×256:

| Blok | Conv kanały | Po MaxPool 2×2 |
|------|-------------|-----------------|
| 1    | 32          | 128×128         |
| 2    | 64          | 64×64           |
| 3    | 128         | 32×32           |
| 4    | 256         | 16×16           |
| 5    | 256         | 8×8             |
| 6    | 256         | 4×4             |
| Flatten | — | 4096 |
| Dense | n_neurons (Optuna) | — |
| Dropout | rate (Optuna) | — |
| Dense | 1 (liniowa) | — |

Wszystko reszta zgodne z papierem: `Conv 3×3 same → BatchNorm → MaxPool 2×2`, ReLU,
MAE loss, Adam, brak aktywacji na wyjściu.

Liczba parametrów: **~3 M** (zostajemy w zakresie modeli lightweight; papier raportuje ~0.9 M
na patchach, my płacimy 3× więcej za pełen kontekst, ale to wciąż 30× mniej niż TReS/DEIQT).


In [ ]:
def build_model(n_neurons: int = 512,
                dropout_rate: float = 0.5,
                eta: float = 1e-3) -> keras.Model:
    """Buduje i kompiluje sieć — architektura z dokumentacji powyżej."""
    conv_filters = [32, 64, 128, 256, 256, 256]   # 6 bloków

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = inp
    for i, f in enumerate(conv_filters):
        x = layers.Conv2D(f, kernel_size=3, padding='same',
                          activation='relu', name=f'conv{i+1}')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}')(x)
        x = layers.MaxPooling2D(pool_size=2, strides=2,
                                padding='valid', name=f'pool{i+1}')(x)
    # po 6 blokach: 256 / 2^6 = 4 → feature map 4×4×256 = 4096

    x = layers.Flatten(name='flatten')(x)
    x = layers.Dense(n_neurons, activation='relu', name='dense_hidden')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    out = layers.Dense(1, activation='linear', name='quality')(x)

    model = keras.Model(inp, out, name='biqa_cnn_full')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=eta),
        loss='mae',                  # papier równ. 5
        metrics=['mae', 'mse'],
    )
    return model


# Przykład — żeby zobaczyć liczbę parametrów
_demo = build_model(n_neurons=512, dropout_rate=0.5, eta=1e-3)
_demo.summary()

## 9. Tuning hiperparametrów — Optuna + TPE

Bayesowska optymalizacja z Tree-structured Parzen Estimator (papier sekcja 3.2, ref. [16]).
Każdy trial: trening do `EPOCHS_TUNING` z early stoppingiem, score = best `val_mae`.

> **Ostrzeżenie:** to drogi krok. Na GPU klasy RTX 4070 (jak w papierze): ~10-15 min per trial.
> 20 triali ≈ 3-5 h. Możesz zmniejszyć `N_OPTUNA_TRIALS` na 5-8 dla szybkiego sanity-check.


In [ ]:
class OptunaPruningCallback(callbacks.Callback):
    """Zgłasza pośrednie wyniki Optunie i ucina słabe triale."""
    def __init__(self, trial, monitor='val_mae'):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is None:
            return
        self.trial.report(float(value), epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()


def objective(trial: optuna.Trial) -> float:
    # Sample hiperparametrów z zakresów Tabeli 2
    n_neurons    = trial.suggest_int  ('n_neurons',    *HP_SPACE['n_neurons'])
    eta          = trial.suggest_float('eta',          *HP_SPACE['eta'],   log=True)
    dropout_rate = trial.suggest_float('dropout_rate', *HP_SPACE['dropout_rate'])

    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_model(n_neurons=n_neurons, dropout_rate=dropout_rate, eta=eta)
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_TUNING,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[
            callbacks.EarlyStopping(monitor='val_mae', patience=PATIENCE,
                                    restore_best_weights=True),
            OptunaPruningCallback(trial, monitor='val_mae'),
        ],
    )
    return float(min(hist.history['val_mae']))


study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f"Start tuningu: {N_OPTUNA_TRIALS} triali")
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print("\n=== Najlepszy trial ===")
print(f"  val_mae       = {study.best_value:.4f}")
print(f"  n_neurons     = {study.best_params['n_neurons']}")
print(f"  eta           = {study.best_params['eta']:.5f}")
print(f"  dropout_rate  = {study.best_params['dropout_rate']:.3f}")

# Zachowaj dla raportu
with open(OUT_DIR / 'best_params.json', 'w') as f:
    json.dump(study.best_params, f, indent=2)

## 10. Finalny trening

Trenujemy najlepszą konfiguracją na pełnej liczbie epok z early stoppingiem na `val_mae`.


In [ ]:
keras.backend.clear_session()
tf.random.set_seed(SEED)

best = study.best_params
model = build_model(
    n_neurons    = best['n_neurons'],
    dropout_rate = best['dropout_rate'],
    eta          = best['eta'],
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_FINAL,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_mae', patience=PATIENCE,
                                restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5,
                                    patience=4, min_lr=1e-6, verbose=1),
    ],
)
print("\n✅ Trening zakończony")

## 11. Ewaluacja — PLCC / SROCC / KROCC

Metryki BIQA z papieru (sekcja 4):

- **SROCC** (równ. 8) — bezpośrednio na surowych predykcjach (jest niezmienniczy na monotoniczne transformacje).
- **PLCC** (równ. 7) — *po* dopasowaniu 5-parametrowej funkcji logistycznej (równ. 6):
$$p(x, \beta) = \beta_1 \left[\tfrac{1}{2} - \tfrac{1}{1 + e^{\beta_2 (x - \beta_3)}}\right] + \beta_4 x + \beta_5$$
- **KROCC** (Kendall tau) — dodatkowo, jak w papierze.


In [ ]:
def logistic_5param(x, b1, b2, b3, b4, b5):
    return b1 * (0.5 - 1.0 / (1.0 + np.exp(b2 * (x - b3)))) + b4 * x + b5


def fit_and_map(y_pred_raw, y_true):
    """Dopasuj 5-param logistic y_pred_raw → y_true, zwróć zmapowane predykcje + parametry."""
    p0 = [np.max(y_true) - np.min(y_true),  # b1
          1.0,                                # b2
          float(np.mean(y_pred_raw)),         # b3
          0.0, float(np.mean(y_true))]        # b4, b5
    try:
        beta, _ = curve_fit(logistic_5param, y_pred_raw, y_true,
                            p0=p0, maxfev=10_000)
    except Exception:
        beta = p0
    return logistic_5param(y_pred_raw, *beta), beta


def evaluate_set(model, X, y_true, name='test'):
    y_pred_raw = model.predict(X, batch_size=BATCH_SIZE, verbose=0).flatten()
    y_pred_fit, _ = fit_and_map(y_pred_raw, y_true)

    plcc , _ = pearsonr (y_pred_fit, y_true)
    srocc, _ = spearmanr(y_pred_raw, y_true)
    krocc, _ = kendalltau(y_pred_raw, y_true)
    mae   = float(np.mean(np.abs(y_pred_raw - y_true)))
    rmse  = float(np.sqrt(np.mean((y_pred_raw - y_true) ** 2)))

    print(f"=== {name.upper()} ===")
    print(f"  PLCC  = {plcc:.4f}   (po dopasowaniu 5-param logistic)")
    print(f"  SROCC = {srocc:.4f}")
    print(f"  KROCC = {krocc:.4f}")
    print(f"  MAE   = {mae:.3f}")
    print(f"  RMSE  = {rmse:.3f}")
    return {
        'plcc': plcc, 'srocc': srocc, 'krocc': krocc,
        'mae': mae, 'rmse': rmse,
        'y_pred_raw': y_pred_raw, 'y_pred_fit': y_pred_fit, 'y_true': y_true,
    }


results_val  = evaluate_set(model, X_val,  y_val,  'val')
print()
results_test = evaluate_set(model, X_test, y_test, 'test')

### 11a. Wyniki w rozbiciu na pary degradacji

Sprawdzamy, czy model nie psuje się systematycznie na jednej parze.


In [ ]:
df_test_eval = df_test.copy()
df_test_eval['y_true']     = results_test['y_true']
df_test_eval['y_pred_raw'] = results_test['y_pred_raw']
df_test_eval['err']        = df_test_eval['y_pred_raw'] - df_test_eval['y_true']

print("Test — MAE per para degradacji × scale_factor:")
print(df_test_eval.groupby(['pair_abbr', 'scale_factor'])['err']
                  .agg(['count', lambda s: np.mean(np.abs(s))])
                  .rename(columns={'<lambda_0>': 'MAE'})
                  .round(2)
                  .to_string())

## 12. Wykresy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Krzywe uczenia
ax = axes[0]
ax.plot(history.history['loss'],     label='train', linewidth=2)
ax.plot(history.history['val_loss'], label='val',   linewidth=2)
ax.set_xlabel('Epoka'); ax.set_ylabel('MAE loss')
ax.set_title('Krzywe uczenia')
ax.legend(); ax.grid(alpha=0.3)

# (b) Scatter pred (raw) vs true + dopasowana logistic
ax = axes[1]
ax.scatter(results_test['y_pred_raw'], results_test['y_true'],
           s=12, alpha=0.4, label='test (raw)')
order = np.argsort(results_test['y_pred_raw'])
ax.plot(results_test['y_pred_raw'][order], results_test['y_pred_fit'][order],
        color='red', linewidth=2, label='5-param logistic fit')
ax.set_xlabel('Predykcja modelu (raw)'); ax.set_ylabel('Pseudo-MOS (true)')
ax.set_title(f"Test: PLCC={results_test['plcc']:.3f}, SROCC={results_test['srocc']:.3f}")
ax.legend(); ax.grid(alpha=0.3)

# (c) Histogram błędu per para
ax = axes[2]
for pair in sorted(df_test_eval['pair_abbr'].unique()):
    sub = df_test_eval[df_test_eval['pair_abbr'] == pair]
    ax.hist(sub['err'], bins=20, alpha=0.5, label=pair)
ax.set_xlabel('błąd predykcji'); ax.set_ylabel('liczba próbek')
ax.set_title('Rozkład błędów per para'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_eval.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Zapis modelu + artefaktów

In [ ]:
# Model
model.save(OUT_DIR / 'biqa_cnn_full.keras')

# Wyniki + best params + historia
artifacts = {
    'best_params': study.best_params,
    'val_metrics':  {k: results_val[k]  for k in ('plcc','srocc','krocc','mae','rmse')},
    'test_metrics': {k: results_test[k] for k in ('plcc','srocc','krocc','mae','rmse')},
    'history':      {k: [float(v) for v in vals] for k, vals in history.history.items()},
}
with open(OUT_DIR / 'artifacts.json', 'w') as f:
    json.dump(artifacts, f, indent=2)

print(f"✅ Zapisane do {OUT_DIR}/:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"   {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

## 14. Inferencja na nowym obrazie

Helper do predykcji jakości dowolnego PNG-a.


In [ ]:
def predict_quality(image_path, model=model) -> float:
    """Zwraca przewidywany pseudo-MOS dla dowolnego obrazu."""
    x = load_image(image_path)[None, ...]   # batch dimension
    return float(model.predict(x, verbose=0)[0, 0])


# Demo — pierwszy obraz testowy
demo_row = df_test.iloc[0]
pred = predict_quality(demo_row['path'])
print(f"Obraz:    {demo_row['sample_id']}")
print(f"Degr.:    {demo_row['pair_abbr']}  sf={demo_row['scale_factor']}")
print(f"True MOS: {demo_row['mos']:.1f}")
print(f"Pred MOS: {pred:.1f}")